[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Ollama_Jev_Gymnasium.ipynb)

**Colab quick start:** Open this notebook with the button above. Connect to a local runtime to use Ollama on your computer, or set `OLLAMA_URL` (and `JEV_URL` when using Jev) to a server reachable from your hosted Colab runtime. Run the cells in order after your server is ready; the button opens the notebook but does not start an Ollama or Jev server.

# Ollama + Simple Jev × Gymnasium
A local notebook for CartPole balance, Pendulum swing-up, and HalfCheetah running.

**Two distinct inference paths:** `ollama` uses `/api/chat` with a JSON schema and generates action labels. `jev` calls Simple Jev's `/v1/classifier`, which scores labels without generating JSON. This notebook does not convert Ollama into the native Jev backend and does not invent confidence values. Ollama now documents output-token logprobs, but they are not used here: reproducing Jev scoring requires the exact prompt/token-label contract and complete allowed-label scores.

Run Jupyter on the same machine as Ollama. Start Ollama, download a model of your choice, and check `ollama list`. A hosted Colab kernel's localhost is **not** your laptop; use a local Jupyter/Colab runtime for localhost access. No model downloads, paid endpoints, training, or repository changes occur automatically.

Start with CartPole. HalfCheetah requires coordinated continuous control; this experiment quantizes each joint torque to five levels. A general language model may perform poorly without task training. The notebook pauses simulated time while waiting for the API: playback is **not evidence of real-time control**.

In [ ]:
%pip install "gymnasium[classic-control,mujoco]>=1.0,<2" requests numpy matplotlib imageio imageio-ffmpeg

## Configuration
Use an exact model ID from your server. Leave `OLLAMA_MODEL` empty to select and print the first installed model. Default: three exploratory seeds and 200 environment steps per episode; use more held-out seeds and full horizons for final claims. CartPole's usual horizon is 500, Pendulum's 200, and HalfCheetah's 1000.

On a headless Linux machine, set `MUJOCO_GL=egl` (GPU) or `osmesa` (with system OSMesa libraries) **before importing MuJoCo**, then restart the kernel if necessary. Desktop Windows usually needs no setting. Disable videos if rendering is unavailable.

In [ ]:
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = ""  # exact name from ollama list; empty = first installed model
JEV_URL = "http://localhost:8000"
JEV_MODEL = "Qwen/Qwen3.5-0.8B"  # exact ID/path used to start your Jev server
ENV_ID = "CartPole-v1"  # also Pendulum-v1 or HalfCheetah-v5
METHODS = ["random", "heuristic", "ollama"]  # add "jev" if that server is running
SEEDS = [0, 1, 2]
MAX_STEPS = 200
ACTION_REPEAT = 1  # same for all methods; repeat >1 changes the control problem
RECORD_VIDEO = True  # first seed for each method
HTTP_TIMEOUT = 120
OLLAMA_THINK = False  # use None to omit for models rejecting this parameter

In [ ]:
import collections, csv, datetime, json, pathlib, platform, time, uuid
import gymnasium as gym
import numpy as np
import requests
import imageio.v2 as imageio

TASKS = {
    "CartPole-v1": {
        "names": ["cart_position_m", "cart_velocity_mps", "pole_angle_rad", "pole_angular_velocity_radps"],
        "goal": "Keep the pole upright and the cart near the center. Angle zero is upright; positive angle leans right. Positive cart position is right. Avoid pole falling or cart leaving track.",
        "joints": ["push"], "levels": {"left": 0, "right": 1},
    },
    "Pendulum-v1": {
        "names": ["cos_angle", "sin_angle", "angular_velocity_radps"],
        "goal": "Swing the pendulum upright and stabilize it at angle zero (cos=1, sin=0), using little torque. Positive torque increases angular velocity.",
        "joints": ["torque"], "levels": {"n2": -2., "n1": -1., "zero": 0., "p1": 1., "p2": 2.},
    },
    "HalfCheetah-v5": {
        "names": ["root_z", "root_pitch", "back_thigh_angle", "back_shin_angle", "back_foot_angle", "front_thigh_angle", "front_shin_angle", "front_foot_angle", "root_x_velocity", "root_z_velocity", "root_pitch_velocity", "back_thigh_velocity", "back_shin_velocity", "back_foot_velocity", "front_thigh_velocity", "front_shin_velocity", "front_foot_velocity"],
        "goal": "Coordinate the six joints to run forward in positive x. Reward is forward velocity minus 0.1 times the sum of squared torques. Build a repeated gait from observation and recent history. No trained gait or future-state simulation is supplied.",
        "joints": ["back_thigh", "back_shin", "back_foot", "front_thigh", "front_shin", "front_foot"],
        "levels": {"n1": -1., "n05": -.5, "zero": 0., "p05": .5, "p1": 1.},
    },
}

class DecisionClient:
    def __init__(self, backend, model, url):
        self.backend, self.model, self.url = backend, model, url.rstrip("/")
        self.http = requests.Session()
        self.server_metadata = {}
        if backend == "ollama":
            r = self.http.get(self.url + "/api/tags", timeout=HTTP_TIMEOUT)
            r.raise_for_status()
            models = r.json().get("models", [])
            if not self.model:
                if not models:
                    raise RuntimeError("No Ollama models installed. Run ollama pull MODEL first.")
                self.model = models[0]["name"]
            self.server_metadata = {"selected_model": next((x for x in models if x.get("name") == self.model or x.get("model") == self.model), None)}
            v = self.http.get(self.url + "/api/version", timeout=HTTP_TIMEOUT)
            if v.ok:
                self.server_metadata["version"] = v.json()
        print(f"{backend}: {self.model} at {self.url}")

    def decide(self, env_id, observation, history, step):
        task = TASKS[env_id]
        obs = np.asarray(observation)
        if obs.shape != (len(task["names"]),) or not np.isfinite(obs).all():
            raise ValueError("Unexpected or nonfinite observation; default environment configuration required.")
        state = {"environment": env_id, "step": step,
                 "observation": dict(zip(task["names"], np.round(obs, 5).tolist())),
                 "recent_history": list(history)}
        schema = {"type": "object", "properties": {j: {"type": "string", "enum": list(task["levels"])} for j in task["joints"]}, "required": task["joints"], "additionalProperties": False}
        t0 = time.perf_counter()
        if self.backend == "ollama":
            body = {"model": self.model, "stream": False, "keep_alive": "10m", "format": schema,
                    "messages": [{"role": "system", "content": task["goal"] + " Select one label per control. Labels map to actions: " + json.dumps(task["levels"]) + ". Return only JSON matching: " + json.dumps(schema)},
                                 {"role": "user", "content": json.dumps(state)}],
                    "options": {"temperature": 0, "seed": 0, "num_predict": 256, "num_ctx": 4096}}
            if OLLAMA_THINK is not None:
                body["think"] = OLLAMA_THINK
            r = self.http.post(self.url + "/api/chat", json=body, timeout=HTTP_TIMEOUT)
            r.raise_for_status()
            raw = r.json()
            choices = json.loads(raw["message"]["content"])
            usage = {k: raw.get(k) for k in ["prompt_eval_count", "eval_count", "load_duration", "total_duration"]}
        else:
            questions = {j: {"type": "choice", "instructions": task["goal"] + " Choose the control for " + j,
                              "criteria": {k: "Apply " + str(v) for k, v in task["levels"].items()}} for j in task["joints"]}
            body = {"model": self.model, "state": state, "questions": questions}
            r = self.http.post(self.url + "/v1/classifier", json=body, timeout=HTTP_TIMEOUT)
            r.raise_for_status()
            raw = r.json()
            choices = {j: raw["answers"][j]["choice"] for j in task["joints"]}
            usage = raw.get("usage", {})
        if not isinstance(choices, dict) or set(choices) != set(task["joints"]):
            raise ValueError("Missing or extra action fields")
        if any(not isinstance(v, str) or v not in task["levels"] for v in choices.values()):
            raise ValueError("Invalid action label")
        values = [task["levels"][choices[j]] for j in task["joints"]]
        action = int(values[0]) if env_id == "CartPole-v1" else np.asarray(values, dtype=np.float32)
        return action, {"latency_ms": (time.perf_counter()-t0)*1000, "choices": choices, "usage": usage, "raw": raw}

def baseline_action(method, env_id, obs, rng):
    task = TASKS[env_id]
    if method == "heuristic":
        if env_id != "CartPole-v1":
            raise ValueError("The heuristic baseline supports only CartPole")
        # Hand-written feedback controller, explicitly separate from model actions.
        x, dx, theta, dtheta = obs
        return int(theta + .25*dtheta + .015*x + .025*dx > 0)
    if method == "zero":
        if env_id == "CartPole-v1":
            raise ValueError("CartPole has no zero-force action")
        return np.zeros(len(task["joints"]), dtype=np.float32)
    if method != "random":
        raise ValueError("Unknown method " + method)
    values = list(task["levels"].values())
    action = rng.choice(values, size=len(task["joints"]))
    return int(action[0]) if env_id == "CartPole-v1" else action.astype(np.float32)

def run_benchmark(env_id, methods, seeds, max_steps, action_repeat=1, record_video=True):
    if env_id not in TASKS or max_steps < 1 or action_repeat < 1 or not seeds or not methods:
        raise ValueError("Check environment, methods, seeds and positive limits")
    if len(set(methods)) != len(methods) or set(methods) - {"ollama", "jev", "random", "zero", "heuristic"}:
        raise ValueError("Unknown or repeated methods")
    if "heuristic" in methods and env_id != "CartPole-v1":
        raise ValueError("Use random/zero baselines for this environment")
    if "zero" in methods and env_id == "CartPole-v1":
        raise ValueError("CartPole has no zero-force action")
    out = pathlib.Path("jev_gym_results") / (datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
    out.mkdir(parents=True)
    config = {"env_id": env_id, "methods": methods, "seeds": list(seeds), "max_steps": max_steps,
              "action_repeat": action_repeat, "gymnasium": gym.__version__, "numpy": np.__version__,
              "python": platform.python_version(), "task": TASKS[env_id], "mode": "synchronous_simulation",
              "ollama_think": OLLAMA_THINK, "history_length": 4, "clients": {}}
    clients = {}
    for method in methods:
        if method in {"ollama", "jev"}:
            client = DecisionClient(method, OLLAMA_MODEL if method == "ollama" else JEV_MODEL,
                                    OLLAMA_URL if method == "ollama" else JEV_URL)
            clients[method] = client
            config["clients"][method] = {"model": client.model, "url": client.url, **client.server_metadata}
    (out/"config.json").write_text(json.dumps(config, indent=2))
    # Warm-up is logged separately and excluded from episode latency/reward.
    for method, client in clients.items():
        env = gym.make(env_id)
        try:
            obs, _ = env.reset(seed=int(seeds[0]))
            _, meta = client.decide(env_id, obs, [], 0)
            (out/(method+"_warmup.json")).write_text(json.dumps(meta, indent=2))
        finally:
            env.close()
    rows = []
    for method in methods:
        for episode, seed in enumerate(seeds):
            video = record_video and episode == 0
            env = gym.make(env_id, render_mode="rgb_array" if video else None)
            writer = None
            history = collections.deque(maxlen=4)
            rng = np.random.default_rng(int(seed)+10000)
            latencies, speeds = [], []
            reward_sum = 0.; steps = calls = failures = 0
            terminated = truncated = False
            status, error = "ok", ""
            started = time.perf_counter()
            trace_path = out/f"{method}_seed{seed}.jsonl"
            try:
                obs, info = env.reset(seed=int(seed))
                if video:
                    writer = imageio.get_writer(str(out/f"{method}_seed{seed}.mp4"), fps=env.metadata.get("render_fps", 30), macro_block_size=1)
                    writer.append_data(env.render())
                with trace_path.open("w") as trace:
                    while steps < max_steps and not (terminated or truncated):
                        before = np.asarray(obs).tolist()
                        try:
                            if method in clients:
                                calls += 1
                                action, meta = clients[method].decide(env_id, obs, history, steps)
                                latencies.append(meta["latency_ms"])
                            else:
                                action = baseline_action(method, env_id, obs, rng)
                                meta = {}
                            if not env.action_space.contains(action):
                                raise ValueError("Action outside environment action space")
                        except Exception as exc:
                            failures += 1
                            raise RuntimeError("Decision failed; no fallback action was applied: " + str(exc)) from exc
                        action_json = np.asarray(action).tolist()
                        step_rewards = []
                        for _ in range(min(action_repeat, max_steps-steps)):
                            obs, reward, terminated, truncated, info = env.step(action)
                            steps += 1
                            reward_sum += float(reward)
                            step_rewards.append(float(reward))
                            if env_id == "HalfCheetah-v5":
                                speeds.append(float(obs[8]))
                            if writer is not None:
                                writer.append_data(env.render())
                            if terminated or truncated:
                                break
                        history.append({"observation": np.round(before, 5).tolist(), "action": action_json, "reward": sum(step_rewards)})
                        trace.write(json.dumps({"step": steps, "before": before, "action": action_json, "after": np.asarray(obs).tolist(), "step_rewards": step_rewards, "terminated": bool(terminated), "truncated": bool(truncated), **meta})+"\n")
                        trace.flush()
                        if steps % 25 == 0:
                            print(f"{env_id} {method} seed={seed}: {steps} steps, reward={reward_sum:.2f}", flush=True)
            except Exception as exc:
                status, error = "failed", repr(exc)
                print(error)
            finally:
                env.close()
                if writer is not None:
                    writer.close()
            row = {"method": method, "seed": int(seed), "status": status, "return": reward_sum, "steps": steps,
                   "terminated": bool(terminated), "truncated": bool(truncated),
                   "capped_by_benchmark": steps >= max_steps and not (terminated or truncated),
                   "api_calls": calls, "decision_failures": failures,
                   "latency_mean_ms": float(np.mean(latencies)) if latencies else None,
                   "latency_p95_ms": float(np.percentile(latencies, 95)) if latencies else None,
                   "mean_forward_speed_mps": float(np.mean(speeds)) if speeds else None,
                   "wall_seconds": time.perf_counter()-started, "error": error}
            rows.append(row)
            with (out/"episodes.csv").open("w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerows(rows)
            print(row)
            if status == "failed":
                print("Stopping this method after failure; partial episode is excluded from success summaries.")
                break
    for client in clients.values():
        client.http.close()
    print("Results:", out.resolve())
    return rows, out

## Run the first experiment
Random actions use the same discrete torque levels as the models. The CartPole heuristic is a hand-written controller, never a silent fallback. Failed episodes are recorded and excluded from successful-return summaries. Warm-up requests are logged separately. Each decision trace contains the raw server response.

In [ ]:
rows, output_dir = run_benchmark(ENV_ID, METHODS, SEEDS, MAX_STEPS, ACTION_REPEAT, RECORD_VIDEO)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Video
successful = [r for r in rows if r["status"] == "ok"]
labels = list(dict.fromkeys(r["method"] for r in successful))
if labels:
    means = [np.mean([r["return"] for r in successful if r["method"] == m]) for m in labels]
    stds = [np.std([r["return"] for r in successful if r["method"] == m]) for m in labels]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, means, yerr=stds, capsize=5)
    ax.set(ylabel="Episode return (mean ± population SD)", title="Exploratory results; successful episodes only")
    plt.show()
for m in METHODS:
    good = sum(r["status"] == "ok" for r in rows if r["method"] == m)
    failed = sum(r["status"] != "ok" for r in rows if r["method"] == m)
    print(m, "successful:", good, "failed:", failed, "planned:", len(SEEDS))
for path in sorted(output_dir.glob("*.mp4")):
    print(path.name)
    display(Video(str(path), embed=True, width=640))

## Running experiment: HalfCheetah
Uncomment the next cell after CartPole works. All six torque decisions are requested together. Native Jev scores each joint question independently; the Ollama path generates a joint JSON object. This is a useful end-to-end comparison, not proof of identical inference semantics. No hand-written gait is hidden in the model policy.

`action_repeat=1` is the default. Larger values reduce API calls but hold torques longer and can hurt stability. HalfCheetah's default environment step is 0.05 simulated seconds: sustained real-time control at this setting would require the whole control loop to fit that budget, which this synchronous benchmark does not enforce.

In [ ]:
# Run a short smoke experiment first; increase max_steps to 1000 for full episodes.
# running_rows, running_dir = run_benchmark(
#     "HalfCheetah-v5", ["random", "zero", "ollama"],
#     seeds=[0, 1, 2], max_steps=200, action_repeat=1, record_video=True,
# )
# for path in sorted(running_dir.glob("*.mp4")):
#     display(Video(str(path), embed=True, width=640))

## Interpretation and next step
- Compare equal seeds, episode limits, observation/history, action levels, and repeat counts. Different model weights, quantization or prompt templates remain confounders; record them when reporting results.
- Reward, forward speed, request latency and failure counts answer different questions. A smooth video can be produced by a very slow policy.
- Three seeds are exploratory. Do not tune on the final test seeds. Compare with a trained PPO/SAC policy before making claims about locomotion quality; that baseline is **not included or claimed tested** here.
- Five torque levels discretize the continuous task. Better running may require trained motion primitives, an RL controller with Jev choosing higher-level behaviors, or task-specific training from expert trajectories.
- HTTP/model errors abort the affected method; no synthetic model decision is substituted. Short capped runs are labeled in CSV.
- Output directories contain configuration, per-episode CSV, per-decision JSONL, warm-up response and videos. Keep them with model/version details.

References: [Simple Jev](https://github.com/vtavakkoli/simple-jev), [Ollama chat API](https://docs.ollama.com/api/chat), [structured output](https://docs.ollama.com/capabilities/structured-outputs), [HalfCheetah](https://gymnasium.farama.org/environments/mujoco/half_cheetah/).

Validation: notebook code and local/mock checks are recorded in the delivery message. No benchmark scores for your Ollama model are pre-filled.